# 📓 Semana 8 · Dia 4 — SCD1 e SCD2 com APPLY CHANGES INTO

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (via notebook com dlt em modo batch) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (SCD) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | dim_cliente SCD2 rodando com histórico |

---


## 📖 Teoria — SCD2 na prática

SCD2 mantém o histórico: a dimensão ganha colunas `valid_from`, `valid_to`, `is_current`. Cada mudança de atributo gera uma nova versão da linha.

O Databricks padroniza com **`APPLY CHANGES INTO`** (dentro do DLT):

```python
@dlt.table
def dim_cliente_scd2():
    return (
        dlt.apply_changes(
            target='dim_cliente_scd2',
            source='stg_clientes',
            keys=['CustomerID'],
            sequence_by='updated_at',
            apply_as_append=False,
            except_column_list=['CustomerID'],
            stored_as_scd_type=2))
```


## 📖 Teoria — SCD1 vs SCD2 no APPLY CHANGES

| Parâmetro | SCD1 | SCD2 |
|---|---|---|
| `stored_as_scd_type` | `1` | `2` |
| `apply_as_append` | `True` (sobrescreve) | `False` (gera versões) |
| Resultado | valor atual | histórico completo |


### 💻 Na prática — Simulando SCD2 fora do DLT

Como o DLT roda em pipeline, vamos demonstrar o mesmo padrão com PySpark (mudança de cidade) para ver o mecanismo.


In [ ]:
# Staging: cliente 12345 mudou de cidade
stg = spark.createDataFrame([
    ("12345", "Ana", "CAMPINAS", "2024-06-01"),
    ("99999", "Maria", "BH", "2024-06-01"),
], ["CustomerID", "nome", "cidade", "updated_at"])
stg.createOrReplaceTempView("stg_clientes")
print("Staging com 1 update + 1 insert")

In [ ]:
# Dimensão SCD2 inicial (na vida real, criada por APPLY CHANGES)
spark.sql("""
CREATE OR REPLACE TABLE workspace.prata.dim_cliente_scd2_demo (
  CustomerID STRING, nome STRING, cidade STRING,
  valid_from DATE, valid_to DATE, is_current BOOLEAN) USING DELTA
""")
spark.sql("""INSERT INTO workspace.prata.dim_cliente_scd2_demo VALUES
  ('12345', 'Ana', 'SP', '2024-01-01', NULL, true),
  ('67890', 'João', 'RJ', '2024-01-01', NULL, true)
""")
print("Dimensão inicial criada.")

In [ ]:
# Aplicar a mudança (SCD2 manual — o mesmo efeito do APPLY CHANGES)
from pyspark.sql.functions import current_date, lit, to_date
atuais = spark.table("workspace.prata.dim_cliente_scd2_demo")
# Fechar a linha atual e abrir a nova versão
spark.sql("UPDATE workspace.prata.dim_cliente_scd2_demo SET valid_to = '2024-06-01', is_current = false WHERE CustomerID = '12345'")
stg.select("CustomerID", "nome", "cidade",
           to_date("updated_at").alias("valid_from"))\
    .withColumn("valid_to", lit(None).cast("date"))\
    .withColumn("is_current", lit(True))\
    .write.mode("append").saveAsTable("workspace.prata.dim_cliente_scd2_demo")
display(spark.sql("SELECT * FROM workspace.prata.dim_cliente_scd2_demo ORDER BY CustomerID, valid_from"))

### 💻 Na prática — APPLY CHANGES no DLT (o padrão oficial)

Dentro de um pipeline DLT (arquivo `.py`), cole o exemplo da teoria e rode — em produção é assim que SCD2 é mantido.


In [ ]:
# ===== workspace_file: pipeline_scd.py =====
import dlt
from pyspark.sql.functions import col, to_date

@dlt.table
def stg_clientes():
    return (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/vol_checkpoints/schema_scd")
        .load("/Volumes/workspace/bronze/vol_landing"))

@dlt.table
@dlt.expect_all_or_drop({"cliente_chave": "CustomerID IS NOT NULL"})
def dim_cliente_scd2():
    return (dlt.apply_changes(
        target="dim_cliente_scd2",
        source="stg_clientes",
        keys=["CustomerID"],
        sequence_by="updated_at",
        apply_as_append=False,
        except_column_list=["CustomerID"],
        stored_as_scd_type=2))
print("Arquivo de pipeline SCD2 (cole no Workspace e rode no DLT).")

> 🎯 **Dica de prova**: DEP: `APPLY CHANGES INTO` com `stored_as_scd_type=2` e `sequence_by` é o padrão para SCD2. Pergunta típica: qual função DLT para SCD2? → apply_changes. SCD1 = stored_as_scd_type 1.


## 🎯 Exercícios de fixação

**1.** Explique o papel de sequence_by no apply_changes.

**2.** O que muda em apply_as_append entre SCD1 e SCD2?

**3.** Crie o SCD1 para a dim_produto (correção de descrição).


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** sequence_by

Define a ordem temporal dos eventos (ex.: updated_at) — o DLT usa para decidir qual mudança é a mais recente e fechar versões na ordem certa.

**2.** apply_as_append

SCD1: True (a mudança sobrescreve o valor atual, sem novas linhas). SCD2: False (gera novas linhas de versão, mantendo histórico).

**3.** SCD1 dim_produto

`apply_changes(target='dim_produto', source='stg_produtos', keys=['StockCode'], sequence_by='updated_at', apply_as_append=True, stored_as_scd_type=1)` — a descrição é sobrescrita.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*